# Evaluation Plots

Vision: 

```
from ocean_emulators.plotting import eval_plots

training_url = ...
prediction_url = ...

eval_plots(training_url, prediction_url)
```

Where

```
def eval_plots(...):
    # Run tests on prediction data
    # Plot all relevant panels in a neat and organized way
```


### Imports

In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import warnings
import os
from xarrayutils.plotting import linear_piecewise_scale

### Utils

In [2]:
# from ocean_emulators.postprocessing import post_processor, prediction_data_test

def manual_v0_fixes(ds_input: xr.Dataset) -> xr.Dataset:
    """Manual fixes for the already existing data (for now only v0.0). This should not be used in the future"""
    # fixes that should be checked and fixes on the input data
    area = xr.open_dataset(os.path.join('/pscratch/sd/s/suryad/data', 'Grid_New.nc'))['area_C'].rename({"xu_ocean": "x", "yu_ocean": "y"})
    # from https://github.com/m2lines/ocean_emulators/issues/17
    dz = xr.DataArray(
        [
            5,
            10,
            15,
            20,
            30,
            50,
            70,
            100,
            150,
            200,
            250,
            300,
            400,
            500,
            600,
            800,
            1000,
            1000,
            1000,
        ],
        dims=["lev"],
    )
    z = xr.DataArray(
        [
            2.5,
            10,
            22.5,
            40,
            65,
            105,
            165,
            250,
            375,
            550,
            775,
            1050,
            1400,
            1850,
            2400,
            3100,
            4000,
            5000,
            6000,
        ],
        dims="lev",
    )
    wetmask = ~np.isnan(ds_input.thetao.isel(time=0).reset_coords(drop=True)).load()
    lon = xr.ones_like(ds_input.y) * ds_input.x
    lat = ds_input.y * xr.ones_like(ds_input.x)
    ds_input = ds_input.assign_coords(
        areacello=area, dz=dz, lev=z, wetmask=wetmask, lon=lon, lat=lat
    )
    # give a dummy commit hash
    ds_input.attrs["m2lines/ocean-emulators_git_hash"] = "dummy"
    return ds_input


# i need to test 2d and 3d separately
def split_2d_3d(ds: xr.Dataset):
    ds_2d = xr.Dataset({v: ds[v] for v in ds.data_vars if "lev" not in ds[v].dims})
    ds_3d = xr.Dataset({v: ds[v] for v in ds.data_vars if "lev" in ds[v].dims})
    return ds_2d, ds_3d


def find_index_for_true(da_bool: xr.DataArray):
    """Find slices along all dimensions within a boolean array that have any True value"""
    # all_dims = da_bool.dims
    all_dims = [
        di for di in ["variable", "time"] if di in da_bool.dims
    ]  # all variables that should be checked for indexers
    # not necessary to check e.g. x,y, lev here
    true_found_index = {}
    for dim in all_dims:
        other_dims = [di for di in da_bool.dims if di != dim]
        test = da_bool.any(other_dims).load()
        index = da_bool[dim].isel({dim: test})
        true_found_index[dim] = index.data
    return true_found_index


def test_nan_consistency(ds: xr.Dataset, name="None"):
    """Test the consistency of nan values in the dataset across variables and time
    (compared to a reference at time=0)."""
    ds = ds.to_array()
    ref = ds.isel(time=0)
    # # make sure the ref data has nans in the same places for all variables
    a = (np.isnan(ref.isel(variable=0)) != np.isnan(ref)).all(["variable"])

    # find the index values for true values in b
    index = find_index_for_true(a)
    if not all(len(v) == 0 for v in index.values()):
        raise ValueError(
            "Found non-matching nan values between variables on the first time step."
        )

    ## make sure that the ref nan pattern is the same as every time step
    b = np.isnan(ref) != np.isnan(ds)

    # find the index values for true values in b
    index = find_index_for_true(b)

    # if they are all length 0 all is good, otherwise raise.
    if not all(len(v) == 0 for v in index.values()):
        raise ValueError(
            f"{name}:Found nonmatching nans compared to first time step in the following indexes {index}"
        )


def input_data_test_deep(ds_input: xr.Dataset):
    """Expensive tests that compute on the entire dataset"""
    ds_nan_test_2d, ds_nan_test_3d = split_2d_3d(ds_input)
    print("2D consistency check")
    test_nan_consistency(ds_nan_test_2d, "2D nan consistency check")

    print("3D consistency check")
    test_nan_consistency(ds_nan_test_3d, "3D nan consistency check")


def input_data_test(ds_input: xr.Dataset, deep=False):
    """Test function to assert the format of the input dataset.
    If `deep` is True, this will run expensive compuation across the entire dataset."""

    expected_data_vars = [
        "thetao",
        "so",
        "uo",
        "vo",
        "zos",
        "hfds",
        "tauvo",
        "tauuo",
        "sithick",
        "siconc",
    ]
    # add the derived mean/std variables
    expected_data_vars_full = []
    for v in expected_data_vars:
        expected_data_vars_full.extend([f"{v}_mean", f"{v}_std"])

    expected_coords = [
        "areacello",
        "dz",
        "x",
        "y",
        "time",
        "lev",
        "lon",
        "lat",
        "wetmask",
    ]
    if not set(ds_input.coords.keys()) == set(expected_coords):
        raise ValueError(
            f"Expected coords {set(expected_coords)} but found {list(set(ds_input.coords.keys()))}"
        )

    expected_sizes = {"x": 360, "y": 180, "lev": 19}
    for di, s in expected_sizes.items():
        if not ds_input.sizes[di] == s:
            raise ValueError(
                f"Expected size ({s}) for dimension {di}, but got {ds_input.sizes[di]}"
            )

    check_attrs = ["m2lines/ocean-emulators_git_hash"]
    for attr in check_attrs:
        if attr not in ds_input.attrs.keys():
            raise ValueError(f"Could not find {attr} in dataset attributes")

    # asser shape of coordinates
    dims_expected_on_coords = {
        "wetmask": ["x", "y", "lev"],
        "areacello": ["x", "y"],
        "lon": ["x", "y"],
        "lat": ["x", "y"],
        "dz": ["lev"],
    }
    for co, expected_dims in dims_expected_on_coords.items():
        if not set(expected_dims) == set(ds_input[co].dims):
            raise ValueError(
                f"Expected dimensions {set(expected_dims)} on {co}, but got {set(ds_input[co].dims)}"
            )

    if deep:
        input_data_test_deep(ds_input)


def post_processor(ds: xr.Dataset, ds_truth: xr.Dataset, n_lev) -> xr.Dataset:
    """Converts the prediction output to an xarray dataset with the same dimensions/variables as input"""
    # Always run the input_data_test in non-deep mode here
    try:
        input_data_test(ds_truth, deep=False)
    except ValueError as e:
        raise ValueError(
            f"Checking the input dataset failed with {e}. Please fix those issues before creating a postprocessed dataset."
        )

    # correct swapped dimensions and warn
    if len(ds.x) == 180 and len(ds.y) == 360:
        ds = ds.rename({"x": "x_i", "y": "y_i"}).rename({"x_i": "y", "y_i": "x"})
        warnings.warn(
            "Swapped x and y dimensions detected. Fixing this now, but should be corrected upstream"
        )

    da = ds["__xarray_dataarray_variable__"]
    variables = ["uo", "vo", "thetao", "so"]
    slices = [slice(i, i + n_lev) for i in range(0, len(variables) * n_lev, n_lev)]
    var_slices = {k: sl for k, sl in zip(variables, slices)}
    variables = {
        k: da.isel(var=sl).rename({"var": "lev"}) for k, sl in var_slices.items()
    }
    variables["zos"] = da.isel(var=-1).squeeze()

    ds_out = xr.Dataset(variables)

    ds_out = ds_out.where(ds_truth.wetmask)

    ## attach all coordinates from input
    ds_out = ds_out.assign_coords({co: ds_truth[co] for co in ds_truth.coords})

    return ds_out


def prediction_data_test(ds_prediction: xr.Dataset, ds_input, n_lev):
    """Testfunction to check post-processed prediction output for format"""
    # TODO: Run the test for the preprocessing data here and warn only if it fails
    # That data should have been checked before training and here we only strictly enforce that things reflect the state of the input data.

    expected_sizes = {"x": 360, "y": 180, "lev": n_lev}
    given_sizes = ds_prediction.sizes
    compare_dims = list(
        set(list(expected_sizes.keys()) + list(given_sizes.keys())) - set(["time"])
    )
    if any(expected_sizes[dim] != given_sizes[dim] for dim in compare_dims):
        raise ValueError(
            f"Input dataset does not have the right sizes. Expected{expected_sizes}, got {given_sizes}"
        )

    # ensure all dimensions have coordinate values
    dims_without_coords = [
        di for di in ds_prediction.dims if di not in ds_prediction.coords
    ]
    if len(dims_without_coords) > 0:
        raise ValueError(
            f"Found the following dimensions without coordinates: {dims_without_coords}"
        )

    # ensure the attributes are the same on both datasets
    # if not ds_prediction.attrs == ds_input.attrs:
    #     raise ValueError(
    #         f"Prediction and Input datasets do not have matching attributes"
    #     )
    # Check that the wetmask is applied to the data
    mask_test = ~np.isnan(ds_prediction.isel(time=0).reset_coords(drop=True)).load()
    if not (mask_test.to_array() == ds_input.wetmask).all():
        raise ValueError(
            "Wetmask does not match between `ds_prediction` and `ds_input`!"
        )

    # TODO: ensure that both arrays have the same coordinates

    # TODO: Check that the wetmask is applied to the data

### Data

In [3]:
import os

levels = 19
ds_input = xr.open_zarr(
    os.path.join("/pscratch/sd/s/suryad/data", "OM4_Horizontal_Regrid_Old.zarr")
)

ds_input = manual_v0_fixes(ds_input)
# our groundtruth is always just a time slice of the training (training is a bad name
ds_groundtruth = ds_input.isel(time=slice(4143, 4743)).isel(lev=slice(None, levels))

# Hist 1
ds_prediction_raw = xr.open_zarr(
    "/pscratch/sd/s/suryad/Ocean_Emulator/Preds/2024-07-26_ConvNextUNetTrain3DEval3DEpochs35Epoch20_Train_global_3D_Test_global_3D_all_N_train_4000_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_4000_rand_seed_1.zarr"
)
hist=1

# Hist 0
# ds_prediction_raw = xr.open_zarr("/pscratch/sd/s/suryad/Ocean_Emulator/Preds/2024-07-04_ConvNext UNet Train3DEval3D100MEpoch25_Train_global_3D_Test_global_3D_all_N_train_4000_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_4000_rand_seed_1.zarr")
# hist=0

ds_prediction = post_processor(
    ds_prediction_raw, ds_groundtruth, levels
) 

# Run the test to make sure the output is formatted correctly
prediction_data_test(ds_prediction, ds_input, levels)

/tmp/ipykernel_1265864/2497544531.py:208: UserWarning: Swapped x and y dimensions detected. Fixing this now, but should be corrected upstream
  warnings.warn(


### Depth Profiles

In [4]:
def profile_mean(ds: xr.Dataset) -> xr.Dataset:
    return ds.weighted(ds.areacello).mean(["x", "y"])


def profile_std(ds: xr.Dataset) -> xr.Dataset:
    return ds.weighted(ds.areacello).std(["x", "y"])

In [6]:
from dask.diagnostics import ProgressBar

with ProgressBar():
    profile_prediction = profile_mean(ds_prediction).load()
    profile_groundtruth = profile_mean(ds_groundtruth).load()
    profile_stdv_prediction = profile_std(ds_prediction).load()
    profile_stdv_groundtruth = profile_std(ds_groundtruth).load()
    
    # KE
    groundtruth_KE = 0.5 * (ds_groundtruth.uo ** 2 + ds_groundtruth.vo ** 2)
    prediction_KE = 0.5 * (ds_prediction.uo ** 2 + ds_prediction.vo ** 2)
    profile_groundtruth = profile_groundtruth.assign(KE=profile_mean(groundtruth_KE).load()*1020)
    profile_prediction = profile_prediction.assign(KE=profile_mean(prediction_KE).load()*1020)
    profile_stdv_groundtruth = profile_stdv_groundtruth.assign(KE=profile_std(groundtruth_KE).load()*1020)
    profile_stdv_prediction = profile_stdv_prediction.assign(KE=profile_std(prediction_KE).load()*1020)

[########################################] | 100% Completed | 20.89 s
[########################################] | 100% Completed | 18.65 s
[########################################] | 100% Completed | 25.58 s
[########################################] | 100% Completed | 21.62 s
[########################################] | 100% Completed | 4.37 sms
[########################################] | 100% Completed | 10.22 s
[########################################] | 100% Completed | 4.79 sms
[########################################] | 100% Completed | 12.38 s


In [6]:
# %load_ext autoreload
# %autoreload 2
%matplotlib inline

In [7]:
def adjust_depth(ax):
    # todo make split vertical axes
    linear_piecewise_scale(1000, 5, ax=ax)
    # indicate the point between the different scalings
    ax.axhline(1000, color="0.5", ls="--")
    # Rearange the yticks
    ax.set_yticks([0, 250, 500, 750, 1000, 3000, 5000])
    

### Depth Profiles

In [ ]:
import matplotlib.colors as colors

def plot_depth_profile(data, title=""):
    plt.clf()
    if 'uo' in title or 'vo' in title:
        gsp = 2
        lsp = 5
    elif 'so' in title:
        gsp = 1
        lsp = 5
    else:
        gsp = 0
        lsp = 4
    
    gain = 10**(-gsp)
    lnrwidth = 10**(1-lsp)
    levels = [-k * 10**-i for i in range(gsp, lsp+1) for k in [5, 1]][1:] + [k*10**-i for i in range(lsp, gsp-1, -1) for k in [1, 5]][:-1] # [-1 * 10**-i for i in range(gsp, lsp+1)] + [10**-i for i in range(lsp, gsp-1, -1)]
    # print(gain, lnrwidth, levels)

    norm = colors.SymLogNorm(linthresh=lnrwidth, linscale=1, vmin=-2*gain, vmax=2*gain, base=10)

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 6))
    pcm = data.plot(
        x="time",
        yincrease=False,
        levels=levels,
        norm=norm,
        cmap='RdBu_r',
        add_colorbar=False,
        ax=ax
    )

    # Adding the colorbar with custom ticks
    cbar = plt.colorbar(pcm, ax=ax, extend='neither')

    # Adjust the plot
    adjust_depth(ax)
    plt.title(title)
    # plt.show()
    plt.savefig(f"../outputs/{title}.png")

In [ ]:
plot_depth_profile(((profile_groundtruth - profile_groundtruth.mean("time"))).thetao, title=f"thetao (truth - mean) hist = {hist}")
plot_depth_profile(((profile_prediction - profile_prediction.mean("time"))).thetao, title=f"thetao (pred - mean) hist = {hist}")
for v in ['KE', 'thetao', 'so', 'uo', 'vo']:
    plot_depth_profile((profile_prediction - profile_groundtruth)[v], title=f"{v} (pred - truth) hist = {hist}")
    

### Timeseries analysis

In [5]:
### Plotting timeseries for each variable for each level
for v in ['thetao', 'so', 'uo', 'vo', 'KE']:
    if not os.path.exists(f"../outputs/{v}_timeseries"):
        os.makedirs(f"../outputs/{v}_timeseries")
    for lev in range(levels):
        plt.clf()
        plt.figure(figsize=[18, 10])
        profile_prediction[v].isel(lev=lev).plot(label='Prediction')
        profile_groundtruth[v].isel(lev=lev).plot(label='Groundtruth')
        min, max = plt.ylim()
        if v == 'thetao':
            plt.ylim(min - 0.25, max + 0.25)
        elif v == 'so':
            plt.ylim(min - 0.2, max + 0.2)
        elif v == 'KE':
            plt.ylim(min - 0.5, max + 0.5)
        plt.xlabel('Time (Days x5)') 
        plt.legend()       
        plt.savefig(f"../outputs/{v}_timeseries/{lev}.png")
        plt.close()

NameError: name 'profile_prediction' is not defined

### OHC

In [ ]:
def raw_ohc(ds):
    c_p = 3850 #J/(kg C) 
    rho_0 = 1025 #kg/m^3
    ohc = ds.thetao * c_p * rho_0 #C*J/(kg C)*kg/m^3 = J/m^3
    return ohc

def vertical_ohc(ds):
    ohc_raw = raw_ohc(ds)
    ohc_intz = ohc_raw.weighted(ds.dz).sum('lev')
    # multiply by area to get Joules
    ohc_intz = ohc_intz * ds.areacello
    return ohc_intz

ohc_truth = vertical_ohc(ds_groundtruth)
ohc_prediction = vertical_ohc(ds_prediction)

ohc_truth_timeseries = ohc_truth.sum(['x','y']).load()
ohc_prediction_timeseries = ohc_prediction.sum(['x','y']).load()

In [ ]:
plt.clf()
plt.figure(figsize=[12, 10])
(ohc_truth_timeseries).plot(label='Ground_truth')
(ohc_prediction_timeseries).plot(label='Prediction')
plt.ylim(2e25, 2.2e25)
plt.title("Integrated OHC")
plt.legend()
plt.savefig("../outputs/ohc_timeseries.png")

In [ ]:
def ohc_map(ohc_intz):
    # return last 5 years - first 5 years
    return ohc_intz.isel(time=slice(-73, None)).mean('time')-ohc_intz.isel(time=slice(0, 73)).mean('time')

ohc_truth_map = ohc_map(ohc_truth).load()
ohc_prediction_map = ohc_map(ohc_prediction).load()

plt.clf()
vmax = 5e19
plt.figure(figsize=[24, 6])
plt.subplot(2,2,1)
ohc_truth_map.plot(vmax=vmax)
plt.title('Truth')

plt.subplot(2,2,2)
ohc_prediction_map.plot(vmax=vmax)
plt.title('Predicted')

plt.subplot(2,2,3)
(ohc_prediction_map - ohc_truth_map).plot(vmax=vmax)
plt.title('Difference (predicted-truth)')
plt.savefig("../outputs/ohc_map.png")

In [ ]:
def ohc_profile(ds):
    ohc_raw = raw_ohc(ds)
    # multiply with dz to get extensive quantity
    ohc = ohc_raw * ohc_raw.dz
    return ohc.weighted(ohc['areacello']).sum(['x', 'y'])

ohc_profile_truth = ohc_profile(ds_groundtruth).load()
ohc_profile_prediction = ohc_profile(ds_prediction).load()

In [ ]:
kwargs = dict(yincrease=False, x='time', vmax = 2.5e22)
plt.clf()
plt.figure(figsize=[24, 10])
plt.subplot(2,2,1)
(ohc_profile_truth-ohc_profile_truth.isel(time=0)).plot(**kwargs)
plt.title('OHC per layer - Truth')

plt.subplot(2,2,2)
(ohc_profile_prediction-ohc_profile_prediction.isel(time=0)).plot(**kwargs)
plt.title('OHC per layer - Predicted')
plt.savefig("../outputs/ohc_profile.png")

### Seasonal maps

In [63]:
def seasonal_composite(ds: xr.Dataset) -> xr.Dataset:
    # maybe use flox?
    return ds.groupby("time.season").mean("time")

In [64]:
from dask.diagnostics import ProgressBar

with ProgressBar():
    seasonal_prediction = seasonal_composite(ds_prediction).load()
    seasonal_groundtruth = seasonal_composite(ds_groundtruth).load()

[########################################] | 100% Completed | 42.77 s
[########################################] | 100% Completed | 21.26 s


In [65]:
plt.clf()
(seasonal_prediction - seasonal_groundtruth).isel(
    lev=[0, 1, 2, 3, 4, 5, 6, 13]
).thetao.plot(col="season", row="lev", robust=True)
plt.savefig("../outputs/seasonal_composite.png")

## More Plots...

### Thetao

In [59]:
plt.clf()
plt.figure(figsize=[12, 5])
ref_std = profile_stdv_groundtruth.mean("time")
((profile_prediction - profile_groundtruth) / ref_std).so.plot(
    x="time", yincrease=False
)
plt.savefig("../outputs/test.png")

### Metrics

In [96]:
mse = ((((ds_prediction - ds_groundtruth) ** 2).mean()) ** (1 / 2)).load()

/srv/conda/envs/notebook/lib/python3.11/site-packages/dask/array/core.py:4832: PerformanceWarning: Increasing number of chunks by factor of 75
  result = blockwise(
/srv/conda/envs/notebook/lib/python3.11/site-packages/dask/array/core.py:4832: PerformanceWarning: Increasing number of chunks by factor of 75
  result = blockwise(
/srv/conda/envs/notebook/lib/python3.11/site-packages/dask/array/core.py:4832: PerformanceWarning: Increasing number of chunks by factor of 75
  result = blockwise(
/srv/conda/envs/notebook/lib/python3.11/site-packages/dask/array/core.py:4832: PerformanceWarning: Increasing number of chunks by factor of 75
  result = blockwise(
/srv/conda/envs/notebook/lib/python3.11/site-packages/dask/array/core.py:4832: PerformanceWarning: Increasing number of chunks by factor of 64
  result = blockwise(


In [97]:
mse

<xarray.Dataset> Size: 40B
Dimensions:  ()
Data variables:
    uo       float64 8B 0.04853
    vo       float64 8B 0.04463
    thetao   float64 8B 0.5248
    so       float64 8B 0.1634
    zos      float64 8B 0.06494